# Quantum ML version of your NDVI project

This notebook uses a quantum classifier (QSVC) from `qiskit-machine-learning` to classify synthetic pixels
as 'healthy' vegetation or not, based on the same synthetic NDVI + precipitation dataset used in the classic version.

Notes:
- You need `qiskit` and `qiskit-machine-learning` installed to run the quantum cells.
- If not installed, run `pip install qiskit qiskit-machine-learning` in your environment (may require internet).

The notebook will reduce features with PCA to 2 components (2 qubits), map them with a feature map, and train a QSVC on a local Aer simulator.


In [ ]:
# 0. (Optional) Install qiskit and qiskit-machine-learning if not available
# Uncomment and run the following line if you need to install packages.
# !pip install qiskit qiskit-machine-learning

# Then restart the kernel and run the notebook cells.

In [ ]:
# 1. Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Quantum imports wrapped in try/except to give a helpful message if packages are missing
try:
    import qiskit
    from qiskit import Aer
    from qiskit.utils import QuantumInstance
    from qiskit.circuit.library import ZZFeatureMap, TwoLocal
    try:
        # QSVC import path (may vary depending on qiskit-machine-learning version)
        from qiskit_machine_learning.algorithms import QSVC
    except Exception:
        from qiskit_machine_learning.algorithms.classifiers import QSVC
    QISKIT_AVAILABLE = True
except Exception as e:
    print('Warning: Qiskit or qiskit-machine-learning not available in this environment.')
    print('Quantum cells will not run. To install, run: pip install qiskit qiskit-machine-learning')
    QISKIT_AVAILABLE = False

%matplotlib inline
sns.set(style='whitegrid')

In [ ]:
# 2. Recreate the synthetic NDVI dataset (same generation logic as classic notebook)
np.random.seed(42)
num_pixels = 800

def make_seasonal(months=12, amplitude=1.0, phase=0.0):
    m = np.arange(months)
    return amplitude * np.sin(2 * np.pi * (m/months) + phase)

lats = np.random.uniform(10.0, 20.0, size=num_pixels)
lons = np.random.uniform(75.0, 85.0, size=num_pixels)
elevation = np.random.uniform(0, 1500, size=num_pixels)
soil_type = np.random.choice([0,1,2], size=num_pixels, p=[0.5,0.3,0.2])
months = 12
records = []
for i in range(num_pixels):
    base_precip = np.interp(lats[i], [10,20], [800,400])
    season = make_seasonal(months, amplitude=0.6, phase=np.random.rand()*2*np.pi)
    for m in range(months):
        precip = max(0, base_precip * (0.5 + 0.8*(season[m]+1)/2) + np.random.normal(0,40))
        ndvi_mean = 0.2 + 0.001*(-elevation[i]) + 0.0015*precip + 0.05*(soil_type[i]==1) - 0.03*(soil_type[i]==2)
        ndvi = np.clip(ndvi_mean + 0.12*season[m] + np.random.normal(0,0.05), -0.1, 0.95)
        records.append({
            'pixel_id': i,
            'lat': lats[i],
            'lon': lons[i],
            'elevation': elevation[i],
            'soil_type': soil_type[i],
            'month': m+1,
            'precip_mm': precip,
            'ndvi': ndvi
        })

df = pd.DataFrame.from_records(records)
df['year'] = 2024

agg = df.groupby('pixel_id').agg(
    mean_ndvi=('ndvi','mean'),
    std_ndvi=('ndvi','std'),
    total_precip=('precip_mm','sum'),
    mean_precip=('precip_mm','mean'),
    elevation=('elevation','first'),
    lat=('lat','first'),
    lon=('lon','first'),
    soil_type=('soil_type','first')
).reset_index()

# Prepare features and target (classification: healthy if mean_ndvi > 0.4)
X = agg[['total_precip','mean_precip','elevation','soil_type','lat','lon','std_ndvi']].copy()
X = pd.get_dummies(X, columns=['soil_type'], prefix='soil')
y = (agg['mean_ndvi'] > 0.4).astype(int)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train/test shapes:', X_train.shape, X_test.shape)

In [ ]:
# 3. Reduce features with PCA to 2 components for 2-qubit quantum encoding
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca = pca.transform(X_test_s)

print('Explained variance ratio (2 PCs):', pca.explained_variance_ratio_.round(4))
print('Sample transformed features (train):')
X_train_pca[:5]

In [ ]:
# 4. Classical baseline: logistic regression on the 2 PCA components
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression()
clf.fit(X_train_pca, y_train)
print('Baseline accuracy (logistic on 2 PCs):', round(accuracy_score(y_test, clf.predict(X_test_pca)), 4))

In [1]:
# 5. Quantum classifier (QSVC) setup and training
if not QISKIT_AVAILABLE:
    print('Qiskit or qiskit-machine-learning not available — cannot run quantum classifier here.')
else:
    # Use ZZFeatureMap to encode 2 features into 2 qubits
    feature_map = ZZFeatureMap(feature_dimension=2, reps=1)
    # Setup backend (Aer simulator) and QuantumInstance
    try:
        backend = Aer.get_backend('aer_simulator')
    except Exception:
        backend = Aer.get_backend('qasm_simulator')
    quantum_instance = QuantumInstance(backend, shots=1024, seed_simulator=42, seed_transpiler=42)

    # Initialize QSVC
    qsvc = QSVC(quantum_instance=quantum_instance, feature_map=feature_map)

    # QSVC expects features in [-1,1] or similar; scale PCA outputs to [-pi, pi] range for phase encoding
    def scale_to_range(X, a=-np.pi, b=np.pi):
        Xmin = X.min(axis=0)
        Xmax = X.max(axis=0)
        # avoid division by zero
        rng = np.where((Xmax - Xmin)==0, 1, Xmax - Xmin)
        return a + (X - Xmin) * (b - a) / rng

    Xtr_q = scale_to_range(X_train_pca)
    Xte_q = scale_to_range(X_test_pca)

    print('Fitting QSVC (this may take a minute on simulator)...')
    qsvc.fit(Xtr_q, y_train)
    y_q_pred = qsvc.predict(Xte_q)
    print('QSVC accuracy:', round(accuracy_score(y_test, y_q_pred), 4))
    print('\nClassification report:\n', classification_report(y_test, y_q_pred))

    cm = confusion_matrix(y_test, y_q_pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion matrix - QSVC')
    plt.show()

NameError: name 'QISKIT_AVAILABLE' is not defined

----

## Notes and troubleshooting

- If you get import errors for `qiskit` or `qiskit-machine-learning`, install them with:

```
pip install qiskit qiskit-machine-learning
```

- On some systems qiskit installation may take a while and require additional system packages.
- If QSVC import path changes across versions, the notebook's try/except will attempt a common alternative import path.

If you want me to instead build a *variational quantum classifier* (VQC) using `TwoLayerQNN` / `EstimatorQNN`, or provide GPU-accelerated simulation (if available), I can create that variant as well.